# 🚀 ViSceT5 — PreSTU SplitOCR Pre-Training trên Kaggle

Notebook hướng dẫn quy trình tiền huấn luyện (**PreSTU SplitOCR**) mô hình **ViSceT5** trên 2 bộ dữ liệu Scene-Text tiếng Việt: **VinText** và **EVJVQA**.

### 📌 Nguyên lý phương pháp PreSTU (Google Research, 2023):
1. **Chỉ nhận Image Pixels + Text Prompt:** Mô hình encoder nhận ảnh điểm ảnh từ CLIP-ViT và chuỗi văn bản tiền tố (prefix prompt).
2. **Cơ chế SplitOCR:** Sắp xếp OCR theo không gian (Top-Left -> Bottom-Right), cắt ngẫu nhiên tại vị trí m in [0, N-1]:
   * m = 0: Pure OCR (Prompt: `"Generate ocr_text in vi:"` -> Target: Toàn bộ N từ OCR).
   * m > 0: Split Continuation (Prompt: `"Generate ocr_text in vi: <OCR_1>...<OCR_m>"` -> Target: `"<OCR_{m+1}>...<OCR_N>"`).
3. **Dual-Target Optimization:** Kết hợp Decoder Text CE Loss và BBox CE Loss trực tiếp qua kiến trúc mô hình đã cố định.

## 1. Clone Codebase & Checkout Nhánh Pretrain

In [ ]:
# Clone repo và checkout đúng nhánh exp/pretrain-gen-all
import os
if not os.path.exists("/kaggle/working/ViSceT5"):
    !git clone https://github.com/Kussssssss/ViSceT5.git /kaggle/working/ViSceT5
%cd /kaggle/working/ViSceT5
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

## 2. Cài Đặt Môi Trường Chuẩn (Khớp hoàn toàn requirements.txt: transformers 4.45.2)

In [ ]:
# Gỡ bản transformers/accelerate mặc định của Kaggle và cài đúng phiên bản requirements.txt
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

## 3. Chuẩn Bị Dữ Liệu (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache.

In [ ]:
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)

In [ ]:
# Khởi tạo mô hình OpenViVQAModel trong tiến trình độc lập
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

* **Epochs:** 10
* **Batch size:** 4 (per device) x 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** 1e-4
* **Output dir:** `/kaggle/working/pretrain_output`

In [ ]:
!python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --num_train_epochs 10 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.0001 \
    --save_total_limit 1 \
    --output_dir /kaggle/working/pretrain_output \
    --logging_dir /kaggle/working/pretrain_output/logs

## 6. Nén Checkpoint Để Tải Về / Upload

In [ ]:
# Nén toàn bộ checkpoint pretrain thành file ZIP để tải về từ giao diện Kaggle
!zip -r /kaggle/working/ViSceT5_PreSTU_Pretrain.zip /kaggle/working/pretrain_output
print("✅ Đã nén thành công checkpoint tại /kaggle/working/ViSceT5_PreSTU_Pretrain.zip")

In [ ]:
# (Tùy chọn) Đăng tải trực tiếp checkpoint lên HuggingFace Hub nếu có Token
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="/kaggle/working/pretrain_output",
#     repo_id="your-username/ViSceT5-PreSTU-Pretrained",
#     repo_type="model",
#     token="your_hf_token_here"
# )